In [ ]:
# Paths (relative to the repository root, so the notebooks run on any machine)
# Expected layout:  <repo>/notebooks/ (this file), <repo>/data/, <repo>/models/,
#                   <repo>/figures/, <repo>/results/, <repo>/feature_importance/
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
FIG_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
FI_DIR = ROOT / "feature_importance"
for _d in (DATA_DIR, MODELS_DIR, FIG_DIR, RESULTS_DIR, FI_DIR):
    _d.mkdir(parents=True, exist_ok=True)


## **XGBoost Regression**

In [1]:
import pandas as pd
import joblib
import numpy as np
import xgboost

from xgboost import XGBRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error
)

from scipy.stats import pearsonr, spearmanr

from sklearn.model_selection import RandomizedSearchCV, KFold

In [2]:
# load your saved datasets:

X_train = pd.read_csv(str(DATA_DIR / "X_train1.csv"))
X_test = pd.read_csv(str(DATA_DIR / "X_test1.csv"))

y_train = pd.read_csv(
    str(DATA_DIR / "y_train1.csv")
).squeeze("columns")

y_test = pd.read_csv(
    str(DATA_DIR / "y_test1.csv")
).squeeze("columns")

In [3]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(21440, 1442)
(5361, 1442)
(21440,)
(5361,)


In [4]:
print(X_train.dtypes.value_counts())

bool       759
float64    483
int64      200
Name: count, dtype: int64


In [5]:
# Creating the baseline XGBoost model

xgb = XGBRegressor(
    random_state=42,
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    objective="reg:squarederror",
    n_jobs=-2
)

In [6]:
# Train the Model

xgb.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [7]:
# Generate the predictions

train_pred = xgb.predict(X_train)
test_pred = xgb.predict(X_test)

In [8]:
# Evaluate 

def evaluate(y_true, y_pred, dataset):
    print(f"\n{dataset}")
    print("-"*30)
    print("R²   :", round(r2_score(y_true, y_pred), 4))
    print("MAE  :", round(mean_absolute_error(y_true, y_pred), 4))
    print("RMSE :", round(np.sqrt(mean_squared_error(y_true, y_pred)), 4))


In [9]:
evaluate(y_train, train_pred, "XGBoost Training")
evaluate(y_test, test_pred, "XGBoost Testing")


XGBoost Training
------------------------------
R²   : 0.8414
MAE  : 0.2402
RMSE : 0.3202

XGBoost Testing
------------------------------
R²   : 0.7326
MAE  : 0.3045
RMSE : 0.4054


In [10]:
# Feature Importance

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgb.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(20)

,Feature,Importance
166,fr_pyridine,0.059577
106,MolLogP,0.034311
85,NHOHCount,0.022498
153,fr_nitrile,0.021702
628,n6HRing,0.021233
613,FilterItLogS,0.020739
144,fr_imidazole,0.020574
137,fr_ester,0.014474
95,NumBridgeheadAtoms,0.013852
619,piPC9,0.012224


In [11]:
# Save the feature importance

importance.to_csv(
    str(FI_DIR / "XGBoost_FeatureImportance1.csv"),
    index=False
)

# **Hyperparameter Tuning**

In [13]:
# Create the parameter search space

param_dist = {
    "n_estimators": [300, 500, 700, 1000],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 5, 7, 9],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3, 0.5]
}

In [14]:
# Create the XGBoost Model

xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-2
)

In [12]:
# CV Splitter

cv_splitter = KFold(n_splits=5, shuffle=True, random_state=42)

In [15]:
# Create the RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=30,
    cv=cv_splitter,
    scoring="r2",
    random_state=42,
    verbose=2,
    n_jobs=-2
)

In [16]:
# Start to Tuning the model

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.6, 0.8, ...], 'gamma': [0, 0.1, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-2
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set

In [17]:
# View the best parameters

print(random_search.best_params_)

{'subsample': 0.9, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 9, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 1.0}


In [18]:
# View the best cross validation score

print(random_search.best_score_)

0.7783880040110703


In [19]:
# Retrieve the best model

best_xgb = random_search.best_estimator_

In [20]:
# Make Predictions

train_pred = best_xgb.predict(X_train)
test_pred = best_xgb.predict(X_test)

# Evalute the optimized XGBoost Model

evaluate(y_train, train_pred, "Optimized XGBoost - Training")
evaluate(y_test, test_pred, "Optimized XGBoost - Testing")


Optimized XGBoost - Training
------------------------------
R²   : 0.9196
MAE  : 0.1646
RMSE : 0.228

Optimized XGBoost - Testing
------------------------------
R²   : 0.7925
MAE  : 0.2575
RMSE : 0.3572


In [21]:
# Save the feature importance

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_xgb.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

importance.head(20)

,Feature,Importance
166,fr_pyridine,0.080814
153,fr_nitrile,0.029134
85,NHOHCount,0.028024
106,MolLogP,0.026869
144,fr_imidazole,0.020962
95,NumBridgeheadAtoms,0.015793
89,NumAliphaticRings,0.015627
613,FilterItLogS,0.013531
619,piPC9,0.010635
0,Charge_complex,0.010065


In [22]:
# Save the file 

importance.to_csv(
    str(FI_DIR / "XGBoost_FeatureImportance1.csv"),
    index=False
)

In [23]:
# the optimized XGBoost model outperform the previous models

# Save the optimized XGBoost Model

import joblib

joblib.dump(
    best_xgb,
    str(MODELS_DIR / "Best_XGBoost_Model1.pkl")
)

['C:\\Users\\user\\Ph.d Work\\Models\\Best_XGBoost_Model1.pkl']

## **Calculate the Metrics**

In [24]:
# Load the Train/Test Dataset

X_train = pd.read_csv(str(DATA_DIR / "X_train1.csv"))
X_test = pd.read_csv(str(DATA_DIR / "X_test1.csv"))

y_train = pd.read_csv(str(DATA_DIR / "y_train1.csv")).squeeze()
y_test = pd.read_csv(str(DATA_DIR / "y_test1.csv")).squeeze()

In [25]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(21440, 1442)
(5361, 1442)
(21440,)
(5361,)


In [26]:
# load the saved best model

best_model = joblib.load(str(MODELS_DIR / "Best_XGBoost_Model1.pkl"))

In [27]:
# Generate the prediction

y_train_pred = best_model.predict(X_train)

y_test_pred = best_model.predict(X_test)

In [28]:
# Calculate R²
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

In [29]:
# Calculate RMSE
train_rmse = root_mean_squared_error(y_train, y_train_pred)
test_rmse = root_mean_squared_error(y_test, y_test_pred)

In [30]:
# Calculate MAE
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

In [31]:
# Calculate Pearson correlation

train_pearson, train_pearson_p = pearsonr(
    y_train,
    y_train_pred
)

test_pearson, test_pearson_p = pearsonr(
    y_test,
    y_test_pred
)

In [32]:
# Calculate Spearman correlation

train_spearman, train_spearman_p = spearmanr(
    y_train,
    y_train_pred
)

test_spearman, test_spearman_p = spearmanr(
    y_test,
    y_test_pred
)

In [33]:
# Create the manuscript table

results = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "R²": [train_r2, test_r2],
    "RMSE": [train_rmse, test_rmse],
    "MAE": [train_mae, test_mae],
    "Pearson r": [train_pearson, test_pearson],
    "Pearson p-value": [train_pearson_p, test_pearson_p],
    "Spearman ρ": [train_spearman, test_spearman],
    "Spearman p-value": [train_spearman_p, test_spearman_p]
})

results = results.round(4)

results

,Dataset,R²,RMSE,MAE,Pearson r,Pearson p-value,Spearman ρ,Spearman p-value
0,Training,0.9196,0.2280,0.1646,0.9616,0.0,0.9596,0.0
1,Testing,0.7925,0.3572,0.2575,0.8916,0.0,0.8872,0.0


In [34]:
# Save the table

results.to_csv(
    str(RESULTS_DIR / "XGBoost_Performance_Table1.csv"),
    index=False
)